In [ ]:
import numpy as np
from astropy.table import Table
from astropy.table import join
from astropy.table import vstack
from astropy.table import Column
import healpy as hp

from astropy.io import fits
import pandas as pd

from astropy.coordinates import SkyCoord
import astropy.units as u

import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import LogNorm
matplotlib.rcParams['figure.dpi'] = 360
matplotlib.rcParams['text.usetex'] = True
matplotlib.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
matplotlib.rcParams.update({'font.size': 14,
                            'axes.titlesize': 16,
                            'axes.labelsize': 14,
                            'xtick.labelsize': 12,
                            'ytick.labelsize': 12,})
import matplotlib.image as mpimg
# plt.style.use('dark_background')
import seaborn as sns
cmap = sns.color_palette('mako_r', as_cmap=True)
cmap

In [ ]:
base = '/global/cfs/cdirs/desi/public/dr1/survey/ops/surveyops/tags/1.0/ops'

In [ ]:
!ls $base

In [ ]:
tiles = Table.read(f'{base}/tiles-main.ecsv', format='ascii.ecsv').to_pandas()
tiles.shape

In [ ]:
tiles = tiles[(tiles['IN_DESI'] == True) &
              (tiles['STATUS'] == 'done') &
              (tiles['PROGRAM'] == 'BRIGHT')]
tiles.shape

In [ ]:
nside = 128
npix = hp.nside2npix(nside)

In [ ]:
tile_radius_deg = 1.605
tile_radius_rad = np.radians(tile_radius_deg)

In [ ]:
def radec_to_vec(ra, dec):
    theta = np.radians(90.0 - dec)
    phi = np.radians(ra)
    return hp.ang2vec(theta, phi)

In [ ]:
tile_count_map = np.zeros(npix, dtype=int)

for ra, dec in zip(tiles['RA'].values, tiles['DEC'].values):
    vec = radec_to_vec(ra, dec)
    pix = hp.query_disc(nside, vec, tile_radius_rad, inclusive=True)
    tile_count_map[pix] += 1

In [ ]:
tile_mask = tile_count_map >= 4

In [ ]:
hp.mollview(tile_count_map, title='N tiles per px', cmap=cmap)
hp.mollview(tile_mask.astype(float), title='Mask: Ntiles $>= 4$', cmap='gray_r')

In [ ]:
hp.mollview(tile_count_map, rot=(120, 0, 0), title='N tiles per px', cmap=cmap)
hp.mollview(tile_mask.astype(float), rot=(120, 0, 0), title='Mask: Ntiles $>= 4$', cmap='gray_r')

In [ ]:
base2 = '/global/cfs/cdirs/desi/public/dr1/vac/dr1/lss/guadalupe/v1.0/LSScats/clustering'

In [ ]:
n = Table.read(f'{base2}/BGS_BRIGHT_N_clustering.dat.fits')
s = Table.read(f'{base2}/BGS_BRIGHT_S_clustering.dat.fits')
df = vstack([n,s]).to_pandas()
df.shape

---------------------------

In [ ]:
nside = 128
npix = hp.nside2npix(nside)

tile_radius_deg = 1.605
tile_radius_rad = np.radians(tile_radius_deg)

min_tiles = 4
min_lss_objects = 1

In [ ]:
tiles = Table.read(f'{base}/tiles-main.ecsv', format='ascii.ecsv').to_pandas()
tiles = tiles[(tiles['IN_DESI'] == True) &
              (tiles['PROGRAM'] == 'BRIGHT') &
              (tiles['STATUS'].isin(['done', 'obsend']))].copy()

tiles.shape

In [ ]:
def radec_to_vec(ra_deg, dec_deg):
    theta = np.radians(90.0 - dec_deg)
    phi = np.radians(ra_deg)
    return hp.ang2vec(theta, phi)

tile_count_map = np.zeros(npix, dtype=np.int32)

In [ ]:
for ra, dec in zip(tiles['RA'].values, tiles['DEC'].values):
    vec = radec_to_vec(ra, dec)
    pix_in_disc = hp.query_disc(nside, vec, tile_radius_rad, inclusive=True)
    tile_count_map[pix_in_disc] += 1

tile_mask = tile_count_map >= min_tiles

In [ ]:
base2 = '/global/cfs/cdirs/desi/public/dr1/vac/dr1/lss/guadalupe/v1.0/LSScats/clustering'

In [ ]:
n = Table.read(f'{base2}/BGS_BRIGHT_N_clustering.dat.fits')
s = Table.read(f'{base2}/BGS_BRIGHT_S_clustering.dat.fits')

df = vstack([n, s]).to_pandas()
df = df[['RA', 'DEC']].dropna().copy()

df.shape

In [ ]:
theta_lss = np.radians(90.0 - df['DEC'].values)
phi_lss   = np.radians(df['RA'].values)

pix_lss = hp.ang2pix(nside, theta_lss, phi_lss)

lss_count_map = np.bincount(pix_lss, minlength=npix)
lss_mask = lss_count_map >= min_lss_objects

In [ ]:
final_mask = tile_mask & lss_mask

In [ ]:
tile_count_plot = np.full(npix, hp.UNSEEN, dtype=float)
tile_count_plot[:] = tile_count_map

hp.mollview(tile_count_plot, rot=(120, 0, 0),
            title='Number of BRIGHT tiles per pixel',
            unit=r'$N_{\mathrm{tiles}}$',)
            # cmap=cmap)
hp.graticule()
# plt.savefig('tiles_per_px.png', dpi=360)
plt.show()

In [ ]:
tile_mask_plot = np.full(npix, hp.UNSEEN, dtype=float)
tile_mask_plot[tile_count_map > 0] = tile_mask[tile_count_map > 0].astype(float)

hp.mollview(tile_mask_plot,
            rot=(120, 0, 0),
            title=r'$N_{\mathrm{tiles}} \geq 4$',
            cmap='gray_r')
hp.graticule()
# plt.savefig('tiles_mora4.png', dpi=360)
plt.show()

In [ ]:
lss_count_plot = np.full(npix, hp.UNSEEN, dtype=float)
has_lss = lss_count_map > 0
lss_count_plot[has_lss] = lss_count_map[has_lss]

hp.mollview(lss_count_plot,
            rot=(120, 0, 0),
            title='LSS object count per pixel',
            unit=r'$N_{\mathrm{obj}}$')
hp.graticule()
# plt.savefig('tiles_lss.png', dpi=360)
plt.show()

In [ ]:
final_mask_plot = np.full(npix, hp.UNSEEN, dtype=float)
valid = tile_mask | lss_mask
final_mask_plot[valid] = final_mask[valid].astype(float)

hp.mollview(final_mask_plot,
            rot=(120, 0, 0),
            title=r'Final mask: $N_{\mathrm{tiles}} \geq 4$ and (LSS occupancy)',
            cmap='gray_r')
hp.graticule()
# plt.savefig('final_mask.png', dpi=360)
plt.show()

In [ ]:
#1
hp.mollview(tile_count_plot,
            rot=(120, 0, 0),
            title='Number of BRIGHT tiles per pixel',
            unit=r'$N_{\mathrm{tiles}}$')
hp.graticule()
plt.title('Number of BRIGHT tiles per pixel', fontsize=18, y=1.03)
plt.savefig('panel_1_tiles.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
#2
hp.mollview(tile_mask_plot,
            rot=(120, 0, 0),
            title=r'$N_{\mathrm{tiles}} \geq 4$',
            cmap='gray_r',
            min=0, max=1,
            cbar=False)
hp.graticule()
plt.title(r'$N_{\mathrm{tiles}} \geq 4$', fontsize=18, y=1.03)
plt.savefig('panel_2_tilemask.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
#3
hp.mollview(lss_count_plot,
            rot=(120, 0, 0),
            title='LSS object count per pixel',
            unit=r'$N_{\mathrm{obj}}$')
hp.graticule()
plt.title('LSS object count per pixel', fontsize=18, y=1.03)
plt.savefig('panel_3_lss.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
#4
hp.mollview(final_mask_plot,
            rot=(120, 0, 0),
            title='Final mask',
            cmap='gray_r',
            min=0, max=1,
            cbar=False)
hp.graticule()
plt.title('Final mask', fontsize=18, y=1.03)
plt.savefig('panel_4_finalmask.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
imgs = [mpimg.imread('panel_1_tiles.png'),
        mpimg.imread('panel_2_tilemask.png'),
        mpimg.imread('panel_3_lss.png'),
        mpimg.imread('panel_4_finalmask.png')]

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 10))

for ax, img in zip(axes, imgs):
    ax.imshow(img)
    ax.axis('off')

plt.subplots_adjust(wspace=0.02)
plt.savefig('pipeline_mask_4columns.png', dpi=360, bbox_inches='tight')
plt.show()